# GRAIN on GIFT-Eval



# Prepare

In [ ]:
from gift_eval.data import Dataset

import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()
gift_eval_path = os.getenv("GIFT_EVAL")

if gift_eval_path:
    gift_eval_path = Path(gift_eval_path)
    dataset_names = []
    for dataset_dir in gift_eval_path.iterdir():
        if dataset_dir.name.startswith("."):
            continue
        if dataset_dir.is_dir():
            freq_dirs = [d for d in dataset_dir.iterdir() if d.is_dir()]
            if freq_dirs:
                for freq_dir in freq_dirs:
                    dataset_names.append(f"{dataset_dir.name}/{freq_dir.name}")
            else:
                dataset_names.append(dataset_dir.name)
    print("Available datasets in GIFT_EVAL:")
    for name in sorted(dataset_names):
        print(f"- {name}")
else:
    print("GIFT_EVAL path not found. Check the .env file.")

## Configure

`PORT_DIR` is the `train_and_inference` directory. This notebook lives in the
gift-eval checkout, so Python cannot see the port's modules by default; they
import each other flatly (`constants`, `dataset`, `estimator`, `model`,
`gift_eval_wrapper`, …), so the whole directory goes on `sys.path`.

`WEIGHTS` is a Lightning `.ckpt`, not a SavedModel directory — whatever
`train.py` printed as `run:`, plus `/best/model.ckpt`.

In [ ]:
# =========================
# Configurable section
# =========================
PORT_DIR     = "/home/oliver1024/Documents/my_pfn/GRAIN"
WEIGHTS      = "/home/oliver1024/Documents/my_pfn/GRAIN/model.ckpt"
MODEL_NAME   = "GRAIN"
ALL_DATASETS = ["m4_yearly"]
TERMS        = ["short"]     # the head is 6 wide; medium/long do not fit

DEVICE       = "cpu"         # "cuda" if a GPU is visible
BATCH_SIZE   = 256
MAX_PLOTS    = 5             # per (dataset, term); 0 disables plotting
SAVE_TXT     = False         # one .txt per (series, window) — 22,974 files for m4_yearly

import os, sys

assert os.path.isdir(PORT_DIR), f"no port directory at {PORT_DIR}"
assert os.path.isfile(os.path.join(PORT_DIR, "gift_eval_wrapper.py")), (
    f"{PORT_DIR} has no gift_eval_wrapper.py — is PORT_DIR pointing at pytorch_version/?"
)
assert os.path.isfile(WEIGHTS), f"no checkpoint at {WEIGHTS}"

if PORT_DIR not in sys.path:
    sys.path.insert(0, PORT_DIR)

print(f"port:    {PORT_DIR}\nweights: {WEIGHTS}")

# Wrapper

Everything model-specific lives in `pytorch_version/gift_eval_wrapper.py` — the
four adaptations GIFT-Eval needs (split at the series end, calendar features from
the `start` Period, float64 cast, quantile-column mapping) are documented there.
This cell only wires it to the harness.

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
np.seterr(all="ignore")

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display

from gluonts.model.evaluation import evaluate_forecasts
from gluonts.time_feature import get_seasonality
from gluonts.ev.metrics import (
    MSE, MAE, MASE, MAPE, SMAPE, MSIS, RMSE, NRMSE, ND,
    MeanWeightedSumQuantileLoss,
)

from gift_eval_wrapper import (
    GIFT_QUANTILE_LEVELS,
    build_gift_predictor,
    load_module,
)

# The repo's own notebooks key results off this file, not off `dataset.freq`.
# `dataset.freq` is the pandas alias ("A-DEC"); the leaderboard and every
# baseline in results/ use the canonical short form ("A"). Mixing them produces
# a `dataset` column that silently matches nothing.
dataset_properties_map = json.load(open("dataset_properties.json"))


def build_metrics():
    
    return [
        MSE(forecast_type="mean"),
        MSE(forecast_type=0.5),
        MAE(forecast_type=0.5),
        MASE(forecast_type=0.5),
        MAPE(forecast_type=0.5),
        SMAPE(forecast_type=0.5),
        MSIS(),
        RMSE(forecast_type="mean"),
        NRMSE(forecast_type="mean"),
        ND(forecast_type=0.5),
        MeanWeightedSumQuantileLoss(quantile_levels=GIFT_QUANTILE_LEVELS),
    ]


def save_series_plot(item_id, input_entry, label_entry, forecast, freq, out_path):
    """Context in blue, median in orange, 10th-90th band shaded — same styling as
    the M3 inference plots. The x axis is steps from the forecast start, not
    dates: M4 yearly runs past pandas' 2262 timestamp bound."""
    ctx = np.asarray(input_entry["target"], dtype=np.float64)[-30:]
    y_true = np.asarray(label_entry["target"], dtype=np.float64)
    median = forecast.quantile("0.5")
    lower, upper = forecast.quantile("0.1"), forecast.quantile("0.9")

    ctx_x = np.arange(-len(ctx), 0)
    fut_x = np.arange(0, len(y_true))

    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.fill_between(fut_x, lower, upper, color="#ffbb78", alpha=0.55, linewidth=0,
                    label="Uncertainty (10th-90th quantiles)")
    ax.plot(ctx_x, ctx, color="#1f77b4", linewidth=1.6, label="Ground Truth")
    ax.plot(fut_x, y_true, color="#1f77b4", linewidth=1.6)
    ax.plot(fut_x, median, color="#ff7f0e", linewidth=1.6,
            label="Prediction (50th quantile)")
    ax.axvline(-0.5, color="0.55", linestyle="--", linewidth=1.0)

    ax.set_title(str(item_id), loc="left", fontweight="bold", fontsize=11)
    ax.set_xlabel(f"steps from forecast start ({freq})")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(True, alpha=0.2, linewidth=0.6)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3,
              frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def write_series_txt(txt_path, series_metrics, fcst, label_entry, freq,
                     season_length, series_idx, window_idx):
    mase_val = series_metrics.get("MASE[0.5]", np.nan)
    smape_val = series_metrics.get("sMAPE[0.5]", np.nan) * 100
    predicted = fcst.quantile("0.5")
    actual = np.asarray(label_entry["target"], dtype=np.float64)[-len(predicted):]
    # Periods, not timestamps: M4 yearly runs past pandas' 2262 bound.
    periods = pd.period_range(start=fcst.start_date, periods=len(predicted))

    with open(txt_path, "w") as f:
        f.write(f"Seasonal MASE:  {mase_val:.4f} (seasonality={season_length})\n")
        f.write(f"SMAPE:          {smape_val:.4f}%\n")
        f.write(f"Series Index:   {series_idx} (Window {window_idx})\n")
        f.write(f"Item ID:        {fcst.item_id}\n\n")
        f.write("=" * 70 + "\nDETAILED PREDICTIONS\n" + "=" * 70 + "\n")
        f.write(f"{'Date':<20} {'Actual':>10} {'Predicted':>10} {'Error':>10} {'Error %':>10}\n")
        f.write("-" * 70 + "\n")
        for p, act, pred in zip(periods, actual, predicted):
            err = pred - act
            err_pct = (err / act * 100) if act != 0 else 0.0
            f.write(f"{str(p):<20} {act:10.2f} {pred:10.2f} {err:10.2f} {err_pct:9.1f}%\n")


print("wrapper ready")

# Evaluation driver

In [ ]:
def evaluate_dataset(module, ds_name, term, model_name, metrics, plots_dir,
                     output_dir, save_txt=SAVE_TXT, max_plots=MAX_PLOTS):
    print(f"\n{'='*70}\nProcessing {ds_name} term={term}\n{'='*70}")

    tmp = Dataset(name=ds_name, term=term, to_univariate=False)
    to_univariate = False if tmp.target_dim == 1 else True
    dataset = Dataset(name=ds_name, term=term, to_univariate=to_univariate)
    season_length = get_seasonality(dataset.freq)

    predictor = build_gift_predictor(
        module,
        prediction_length=dataset.prediction_length,
        batch_size=BATCH_SIZE,
        device=DEVICE,
    )
    print(f"freq={dataset.freq}  h={dataset.prediction_length}  "
          f"windows={dataset.windows}  seasonality={season_length}")

    forecasts = list(predictor.predict(dataset.test_data.input))
    label_list = list(dataset.test_data.label)
    input_list = list(dataset.test_data.input)
    n_windows = dataset.test_data.windows

    # Order is load-bearing: evaluate_forecasts zips forecasts against test_data
    # positionally. The transformation emits exactly one instance per entry, so a
    # length mismatch means something upstream dropped a series.
    assert len(forecasts) == len(label_list), (
        f"{len(forecasts)} forecasts vs {len(label_list)} labels"
    )

    eval_kwargs = dict(
        forecasts=forecasts,
        test_data=dataset.test_data,
        metrics=metrics,
        batch_size=BATCH_SIZE,
        mask_invalid_label=True,
        allow_nan_forecast=False,
        seasonality=season_length,
    )
    eval_detailed = (evaluate_forecasts(axis=1, **eval_kwargs)
                     .reset_index(drop=True).to_dict(orient="records"))
    eval_overall = (evaluate_forecasts(axis=None, **eval_kwargs)
                    .reset_index(drop=True).to_dict(orient="records"))

    # Key and domain come from dataset_properties.json, matching every baseline
    # in results/ and the leaderboard. Using dataset.freq here would emit
    # "m4_yearly/A-DEC/short" and join against nothing.
    ds_key = ds_name.split("/")[0] if "/" in ds_name else ds_name
    props = dataset_properties_map[ds_key]
    ds_config = f"{ds_key}/{props['frequency']}/{term}"
    domain = props["domain"]
    num_variates = int(dataset.target_dim)

    overall_row = {
        "dataset": ds_config,
        "model": model_name,
        **{f"eval_metrics/{k}": v for k, v in eval_overall[0].items()},
        "domain": domain,
        "num_variates": num_variates,
    }

    tag = f"{ds_name.replace('/', '_')}_{term}"
    ds_plot_dir = os.path.join(plots_dir, tag)
    if max_plots:
        os.makedirs(ds_plot_dir, exist_ok=True)
    txt_dir = os.path.join(output_dir, "detailed_metrics", tag)
    if save_txt:
        os.makedirs(txt_dir, exist_ok=True)

    detailed_rows = []
    for i, (series_metrics, fcst, label_entry) in enumerate(
        zip(eval_detailed, forecasts, label_list)
    ):
        detailed_rows.append({
            "dataset": ds_config,
            "series_index": i // n_windows,
            "window_index": i % n_windows,
            "model": model_name,
            **{f"eval_metrics/{k}": v for k, v in series_metrics.items()},
            "domain": domain,
            "num_variates": num_variates,
        })
        if save_txt:
            write_series_txt(
                os.path.join(txt_dir, f"{i}.txt"), series_metrics, fcst,
                label_entry, dataset.freq, season_length,
                i // n_windows, i % n_windows,
            )
        if i < max_plots:
            safe_id = str(fcst.item_id).replace("/", "_").replace("\\", "_")
            save_series_plot(
                fcst.item_id, input_list[i], label_entry, fcst, dataset.freq,
                os.path.join(ds_plot_dir, f"{safe_id}.png"),
            )

    print(f"✅ {ds_config}: {len(eval_detailed) // n_windows} series")
    print(f"   MASE[0.5]={overall_row.get('eval_metrics/MASE[0.5]'):.4f}   "
          f"mean_wQL={overall_row.get('eval_metrics/mean_weighted_sum_quantile_loss'):.4f}")
    return overall_row, detailed_rows


def run(weights: str, model_name: str, all_datasets: list, terms=TERMS):
    output_dir = f"../results/{model_name}"
    plots_dir = os.path.join(output_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    module = load_module(weights, device=DEVICE)
    metrics = build_metrics()

    overall_rows, detailed_rows = [], []
    for ds_name in all_datasets:
        for term in terms:
            overall_row, rows = evaluate_dataset(
                module=module, ds_name=ds_name, term=term,
                model_name=model_name, metrics=metrics,
                plots_dir=plots_dir, output_dir=output_dir,
            )
            overall_rows.append(overall_row)
            detailed_rows.extend(rows)

    overall_path = os.path.join(output_dir, "all_results.csv")
    detailed_path = os.path.join(output_dir, "all_results_detailed.csv")

    overall_df = pd.DataFrame(overall_rows)
    if not overall_df.empty:
        overall_df.sort_values(by="dataset").to_csv(overall_path, index=False)
    detailed_df = pd.DataFrame(detailed_rows)
    if not detailed_df.empty:
        detailed_df.sort_values(by=["dataset", "series_index", "window_index"]).to_csv(
            detailed_path, index=False
        )

    print("\n" + "=" * 70)
    print("ALL DATASETS COMPLETE")
    print("=" * 70)
    print(f"📊 Overall:  {overall_path}")
    print(f"📊 Detailed: {detailed_path}")
    print(f"🖼️  Plots:    {plots_dir}")
    return overall_df, detailed_df

# Run

In [ ]:
overall_df, detailed_df = run(WEIGHTS, MODEL_NAME, ALL_DATASETS)
overall_df